# Windmill

Notebook for wind-to-energy calculations.

In [2]:
import os

from collections import defaultdict

from getpass import getpass



import requests



# Windmill model:

# For each 1 m/s wind speed (hourly observation), produce 1 kWh.



YEAR = 2025

STATION_ID = "06180"  # Example station, change if needed



DMI_API_KEY = os.getenv("DMI_API_KEY")

if not DMI_API_KEY:

    DMI_API_KEY = getpass("Enter your DMI API key: ")



if not DMI_API_KEY:

    raise ValueError("No API key provided. Set DMI_API_KEY or enter it when prompted.")



url = "https://dmigw.govcloud.dk/v2/metObs/collections/observation/items"

params = {

    "api-key": DMI_API_KEY,

    "stationId": STATION_ID,

    "parameterId": "wind_speed",

    "datetime": f"{YEAR}-01-01T00:00:00Z/{YEAR}-12-31T23:59:59Z",

    "limit": 20000,

}



response = requests.get(url, params=params, timeout=30)

response.raise_for_status()

payload = response.json()



month_kwh = defaultdict(float)

for feature in payload.get("features", []):

    properties = feature.get("properties", {})

    wind_ms = properties.get("value")

    observed = properties.get("observed") or properties.get("from")

    if wind_ms is None or not observed:

        continue



    month_number = int(observed[5:7])

    month_kwh[month_number] += float(wind_ms)



months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]



print(f"Wind = kWh monthly output for {YEAR} (station {STATION_ID})")

print("-" * 46)

for month_index, month_name in enumerate(months, start=1):

    print(f"{month_name}: {month_kwh[month_index]:.1f} kWh")


Wind = kWh monthly output for 2025 (station 06180)
----------------------------------------------
Jan: 0.0 kWh
Feb: 0.0 kWh
Mar: 0.0 kWh
Apr: 0.0 kWh
May: 0.0 kWh
Jun: 0.0 kWh
Jul: 0.0 kWh
Aug: 10395.9 kWh
Sep: 20923.2 kWh
Oct: 24580.3 kWh
Nov: 19774.4 kWh
Dec: 22451.1 kWh


In [20]:
# Windmill production with random monthly wind speeds (custom m/s -> kWh table)



import random



location = {

    "latitude": 57.30471773730597,

    "longitude": 10.858219550129444,

}



month_order = ["Jan", "Feb", "Mar", "Apr", "Maj", "Jun", "Jul", "Aug", "Sep", "Okt", "Nov", "Dec"]



# Your production table (m/s -> kWh)

wind_to_kwh = {

    1: 0.0,

    2: 0.0,

    3: 1.2,

    4: 7.2,

    5: 14.5,

    6: 24.7,

    7: 37.9,

    8: 58.7,

    9: 74.8,

    10: 85.1,

    11: 90.2,

    12: 94.7,

}



# Random wind speed per month (integer values from 1 to 12 m/s)

monthly_wind_ms = {month: random.randint(1, 12) for month in month_order}



# Calculate monthly production using your table

monthly_production = {}

for month in month_order:

    wind_ms = monthly_wind_ms[month]

    monthly_production[month] = wind_to_kwh[wind_ms]



yearly_total_kwh = sum(monthly_production.values())



print("Vindmølle lokation:", f"{location['latitude']}, {location['longitude']}")

print("Tilfældig vind pr. måned (m/s):")

print("-" * 44)

for month in month_order:

    print(f"{month}: {monthly_wind_ms[month]} m/s")



print("\nMånedlig strømproduktion (kWh):")

print("-" * 44)

for month in month_order:

    print(f"{month}: {monthly_production[month]:.1f} kWh")



print("-" * 44)

print(f"Samlet produktion for året: {yearly_total_kwh:.1f} kWh")


Vindmølle lokation: 57.30471773730597, 10.858219550129444
Tilfældig vind pr. måned (m/s):
--------------------------------------------
Jan: 3 m/s
Feb: 3 m/s
Mar: 3 m/s
Apr: 3 m/s
Maj: 10 m/s
Jun: 3 m/s
Jul: 8 m/s
Aug: 12 m/s
Sep: 8 m/s
Okt: 11 m/s
Nov: 1 m/s
Dec: 11 m/s

Månedlig strømproduktion (kWh):
--------------------------------------------
Jan: 1.2 kWh
Feb: 1.2 kWh
Mar: 1.2 kWh
Apr: 1.2 kWh
Maj: 85.1 kWh
Jun: 1.2 kWh
Jul: 58.7 kWh
Aug: 94.7 kWh
Sep: 58.7 kWh
Okt: 90.2 kWh
Nov: 0.0 kWh
Dec: 90.2 kWh
--------------------------------------------
Samlet produktion for året: 483.6 kWh


In [24]:
# Daily average wind for all 365 days in 2025 from DMI Free Data (metObs) + kWh calculation

# Documentation: https://www.dmi.dk/friedata/dokumentation/meteorological-observations-data



from collections import defaultdict

from datetime import date, datetime, timedelta



import requests



YEAR = 2025

STATION_ID = "06180"  # Change station if needed



url = "https://dmigw.govcloud.dk/v2/metObs/collections/observation/items"



# Fetch month by month to ensure full-year coverage (avoids truncation from a single large request)

all_features = []

for month in range(1, 13):

    month_start = date(YEAR, month, 1)

    if month == 12:

        next_month_start = date(YEAR + 1, 1, 1)

    else:

        next_month_start = date(YEAR, month + 1, 1)



    start_iso = f"{month_start.isoformat()}T00:00:00Z"

    end_iso = f"{(next_month_start - timedelta(days=1)).isoformat()}T23:59:59Z"



    params = {

        "stationId": STATION_ID,

        "parameterId": "wind_speed",

        "datetime": f"{start_iso}/{end_iso}",

        "limit": 20000,

    }



    response = requests.get(url, params=params, timeout=30)

    response.raise_for_status()

    payload = response.json()

    all_features.extend(payload.get("features", []))



# Build daily average wind speed (m/s)

daily_sum = defaultdict(float)

daily_count = defaultdict(int)



for feature in all_features:

    properties = feature.get("properties", {})

    wind_ms = properties.get("value")

    observed = properties.get("observed") or properties.get("from")

    if wind_ms is None or not observed:

        continue



    day = observed[:10]  # YYYY-MM-DD

    daily_sum[day] += float(wind_ms)

    daily_count[day] += 1



# Build all 365 dates in 2025

start_day = date(YEAR, 1, 1)

end_day = date(YEAR, 12, 31)

all_days_2025 = []

cursor = start_day

while cursor <= end_day:

    all_days_2025.append(cursor.isoformat())

    cursor += timedelta(days=1)



daily_avg_ms = {}

for day in all_days_2025:

    if daily_count[day] > 0:

        daily_avg_ms[day] = daily_sum[day] / daily_count[day]

    else:

        daily_avg_ms[day] = None



# Your production table (m/s -> kWh)

wind_to_kwh = {

    1: 0.0,

    2: 0.0,

    3: 1.2,

    4: 7.2,

    5: 14.5,

    6: 24.7,

    7: 37.9,

    8: 58.7,

    9: 74.8,

    10: 85.1,

    11: 90.2,

    12: 94.7,

}



def kwh_from_wind_ms(wind_ms: float) -> float:

    if wind_ms <= 1:

        return wind_to_kwh[1]

    if wind_ms >= 12:

        return wind_to_kwh[12]

    lower = int(wind_ms)

    upper = lower + 1

    lower_kwh = wind_to_kwh[lower]

    upper_kwh = wind_to_kwh[upper]

    fraction = wind_ms - lower

    return lower_kwh + (upper_kwh - lower_kwh) * fraction



daily_kwh = {}

for day in all_days_2025:

    avg_ms = daily_avg_ms[day]

    if avg_ms is None:

        daily_kwh[day] = None

    else:

        daily_kwh[day] = kwh_from_wind_ms(avg_ms)



yearly_total_kwh = sum(value for value in daily_kwh.values() if value is not None)

days_with_data = sum(1 for value in daily_avg_ms.values() if value is not None)



print(f"Daily average wind for all 365 days in {YEAR} (station {STATION_ID})")

print("-" * 72)

for day in all_days_2025:

    if daily_avg_ms[day] is None:

        print(f"{day}: no data")

    else:

        print(f"{day}: {daily_avg_ms[day]:.2f} m/s -> {daily_kwh[day]:.2f} kWh")



print("-" * 72)

print(f"Total days in year: {len(all_days_2025)}")

print(f"Days with data: {days_with_data}")

print(f"Estimated total production for {YEAR}: {yearly_total_kwh:.2f} kWh")


Daily average wind for all 365 days in 2025 (station 06180)
------------------------------------------------------------------------
2025-01-01: 12.21 m/s -> 94.70 kWh
2025-01-02: 5.50 m/s -> 19.57 kWh
2025-01-03: 8.04 m/s -> 59.28 kWh
2025-01-04: 4.64 m/s -> 11.88 kWh
2025-01-05: 6.08 m/s -> 25.76 kWh
2025-01-06: 9.67 m/s -> 81.69 kWh
2025-01-07: 11.05 m/s -> 90.41 kWh
2025-01-08: 10.30 m/s -> 86.65 kWh
2025-01-09: 2.84 m/s -> 1.01 kWh
2025-01-10: 4.85 m/s -> 13.40 kWh
2025-01-11: 8.97 m/s -> 74.35 kWh
2025-01-12: 5.72 m/s -> 21.89 kWh
2025-01-13: 5.16 m/s -> 16.09 kWh
2025-01-14: 7.15 m/s -> 41.09 kWh
2025-01-15: 2.99 m/s -> 1.19 kWh
2025-01-16: 5.01 m/s -> 14.59 kWh
2025-01-17: 5.34 m/s -> 18.00 kWh
2025-01-18: 3.30 m/s -> 2.98 kWh
2025-01-19: 1.42 m/s -> 0.00 kWh
2025-01-20: 3.73 m/s -> 5.55 kWh
2025-01-21: 4.97 m/s -> 14.31 kWh
2025-01-22: 5.72 m/s -> 21.83 kWh
2025-01-23: 7.04 m/s -> 38.81 kWh
2025-01-24: 7.59 m/s -> 50.15 kWh
2025-01-25: 4.71 m/s -> 12.36 kWh
2025-01-26: 1.99 m/